### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [2]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [3]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [4]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [7]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4.1-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4.1-nano")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [8]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [10]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [11]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some pros of choosing AutoGen for your AI Agent project:

1. **Efficiency and Scalability**: AutoGen simplifies the development process for AI agents, allowing for easier orchestration of multiple agents, which can improve overall efficiency and scalability in applications.

2. **Ease of Use**: Built in TypeScript, the framework is designed to be user-friendly, enabling developers to quickly structure and deploy AI agents without getting bogged down in complexity.

3. **Collaborative Capabilities**: AutoGen facilitates communication and collaboration between AI agents, allowing them to work together more effectively, which can lead to improved problem-solving and reasoning.

4. **Multi-Model Support**: You can easily integrate different models (like OpenAI and Claude), enabling flexibility and the ability to leverage various strengths of different AI models.

5. **Built-in Features**: The framework includes helpful features such as retry logic, caching, model fallback, and goal setting, which streamline application development and enhance functionality.

6. **Handling Ambiguity and Feedback**: AutoGen is adept at managing ambiguity and can process feedback effectively, making it a robust choice for complex applications.

7. **Advanced Tool Integration**: It allows for integration with platforms like Azure, which can extend the capabilities of your applications and drive more business value.

These advantages make AutoGen a compelling option for developing AI agents in a structured and efficient manner. 

TERMINATE

## Cons of AutoGen:
Here are some reasons against choosing AutoGen for your new AI Agent project:

1. Documentation Challenges: AutoGen's documentation can be hard to read and lacks sufficient examples, which can make onboarding and effective use difficult.
2. Smaller Ecosystem: Compared to competitors like LangChain, AutoGen has a relatively new and smaller ecosystem, which may limit community support and available integrations.
3. No Visual/No-Code Builder: AutoGen does not provide a visual builder or no-code interface, which can be a drawback for teams wanting easier or more intuitive design options.
4. Complexity and Over-Engineering: AutoGen is designed for multi-agent cooperation, which can lead to over-engineered and complex systems if not carefully managed.
5. Cost and Performance Concerns: Multi-agent workflows can be expensive and may run into rate limits quickly, potentially impacting scalability and cost-effectiveness.
6. Reasoning Challenges and Coordination Issues: Multi-agent setups may suffer from issues like agents getting stuck waiting for approvals or spreading misinformation between agents.

These cons suggest that if your project requires simplicity, strong documentation, or visual tools, or if you want to avoid potential cost and complexity issues, AutoGen might not be the best choice. 

Let me know if you want a comparison with other frameworks or more details!



## Decision:

I recommend against using AutoGen for this project. While it offers significant advantages in scalability, collaboration, and multi-model support, the drawbacks—particularly concerning documentation quality, ecosystem size, complexity, and potential costs—pose substantial risks. Given these considerations, a more mature and well-supported framework might better suit our needs to ensure smoother development and maintenance. TERMINATE

In [12]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [13]:
await host.stop()